# Huấn luyện YOLOv11 (Electronic Components Dataset) trên Google Colab cho FPGA 138K Pro
Notebook này được cấu hình sẵn để giải nén và huấn luyện mô hình YOLOv11 với dataset `Electronic Components.v2i.yolov11.zip` trên môi trường Google Colab.
**Lưu ý quan trọng:** Hãy chắc chắn bạn đã đổi runtime sang **GPU** (vào *Runtime* -> *Change runtime type* -> *T4 GPU* / *L4 GPU*).

### Bước 1: Kết nối Google Drive
Chạy ô dưới đây để cấp quyền cho Colab truy cập vào Google Drive của bạn (nơi bạn tải file `Electronic Components.v2i.yolov11.zip` lên).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Bước 2: Cài đặt Ultralytics và giải nén Dataset
Tải thư viện Ultralytics và giải nén file `Electronic Components.v2i.yolov11.zip` từ Drive vào bộ nhớ NVMe tốc độ cao của Colab (`/content/dataset/electronic_components`).
*Mẹo: Nếu bạn để file zip ở trong thư mục con trên Drive (ví dụ `MyDrive/Datasets/...`), hãy sửa lại đường dẫn trong lệnh `unzip` bên dưới cho khớp.*

In [ ]:
!pip install -q ultralytics
!mkdir -p /content/dataset/electronic_components
!unzip -q "/content/drive/MyDrive/Electronic Components.v2i.yolov11.zip" -d /content/dataset/electronic_components
print("Giải nén dataset thành công!")

### Bước 3: Cập nhật đường dẫn trong `data.yaml`
Dataset từ Roboflow thường để đường dẫn dạng tương đối (`../train/images`), điều này có thể gây lỗi không tìm thấy ảnh khi train trên Colab. Đoạn code dưới đây sẽ tự động cập nhật lại thành đường dẫn chuẩn trên Colab.

In [ ]:
import yaml

yaml_path = '/content/dataset/electronic_components/data.yaml'

with open(yaml_path, 'r', encoding='utf-8') as f:
    data = yaml.safe_load(f)

# Thiết lập thư mục gốc tuyệt đối của dataset trên Colab
data['path'] = '/content/dataset/electronic_components'
# Chuẩn hóa các đường dẫn tập train, val, test
data['train'] = 'train/images'
data['val'] = 'valid/images'
data['test'] = 'test/images'

with open(yaml_path, 'w', encoding='utf-8') as f:
    yaml.dump(data, f, allow_unicode=True)

print("Đã cấu hình lại data.yaml thành công:")
print("-" * 40)
print(yaml.dump(data, allow_unicode=True))

### Bước 4: Huấn luyện có Backup tự động & Trích xuất mô hình (Export sang ONNX)
- **Cơ chế chống mất dữ liệu khi mất điện/rớt mạng:** Toàn bộ quá trình train (weights `last.pt`, `best.pt`) được lưu thẳng vào **Google Drive** (`/content/drive/MyDrive/electronic_components_yolo`). Nếu đang train giữa chừng mà bị mất điện hoặc ngắt kết nối Colab, lần sau bật lại và chạy đúng ô này, code sẽ **tự động khôi phục (Resume)** chạy tiếp từ epoch bị ngắt quãng chứ không train lại từ đầu!
- **Batch size:** Đặt `batch=64` (cao hơn theo yêu cầu, ảnh 416x416 rất nhẹ nên GPU T4/L4 xử lý cực nhanh và mượt mà).

In [ ]:
import os
from ultralytics import YOLO

# 1. Định nghĩa thư mục lưu trữ trên Google Drive để CHỐNG MẤT DỮ LIỆU khi mất điện/rớt mạng Colab
project_dir = '/content/drive/MyDrive/electronic_components_yolo'
run_name = 'train_416'
last_checkpoint = f"{project_dir}/{run_name}/weights/last.pt"

# 2. Kiểm tra xem trước đó có đang train dở dang (bị ngắt kết nối do mất điện/hết timeout) hay không
if os.path.exists(last_checkpoint):
    print(f"🔄 Phát hiện checkpoint cũ tại Google Drive: {last_checkpoint}")
    print("🚀 Đang tự động khôi phục (RESUME) quá trình huấn luyện từ epoch bị gián đoạn...")
    model = YOLO(last_checkpoint)
    results = model.train(resume=True)
else:
    print("🌟 Bắt đầu huấn luyện mới từ đầu với mô hình YOLOv11 Nano...")
    model = YOLO('yolo11n.pt')
    results = model.train(
        data='/content/dataset/electronic_components/data.yaml',
        epochs=100,
        imgsz=416,           # Kích thước 416x416 tối ưu tốc độ inference trên FPGA
        batch=64,            # Tăng batch lên 64 giúp tận dụng VRAM GPU và train nhanh hơn
        workers=8,           # Sử dụng 8 luồng CPU của Colab để đọc dữ liệu siêu nhanh
        amp=True,            # Kích hoạt Mixed Precision (FP16) giúp tăng tốc train
        save=True,           # Tự động lưu checkpoint sau mỗi epoch
        project=project_dir, # LƯU THẲNG VÀO GOOGLE DRIVE: Vĩnh viễn không mất dữ liệu khi mất điện/rớt mạng
        name=run_name
    )

# 3. Export sang định dạng ONNX tối ưu cho FPGA
onnx_path = model.export(format='onnx', imgsz=416, opset=12, simplify=True)

print("=" * 60)
print(f"HOÀN TẤT! File ONNX đã được tạo tại: {onnx_path}")
print("Bạn hãy mở Google Drive của bạn ra, vào thư mục: electronic_components_yolo -> train_416 -> weights/")
print("Tải file best.onnx và best.pt về máy để chuẩn bị bước tiếp theo cho FPGA!")

### Bước 5: Lệnh tiện ích - Sao chép toàn bộ kết quả về Google Drive (Backup thủ công)
Lệnh dưới đây dùng trong trường hợp bạn chạy code phiên bản cũ (dữ liệu đang nằm ở `/content/runs/`) hoặc muốn sao chép toàn bộ mọi kết quả train đang có trên máy ảo Colab về một thư mục duy nhất trên Google Drive để lưu trữ lâu dài.

In [ ]:
# Tạo thư mục backup trên Google Drive
!mkdir -p "/content/drive/MyDrive/FOD_YOLO_Backup_All"

# Copy toàn bộ thư mục runs (và cả thư mục project cũ nếu có) sang Google Drive
!cp -r /content/runs/* "/content/drive/MyDrive/FOD_YOLO_Backup_All/" 2>/dev/null || true
!cp -r /content/electronic_components_yolo/* "/content/drive/MyDrive/FOD_YOLO_Backup_All/" 2>/dev/null || true

print("=" * 60)
print("✅ Đã sao chép toàn bộ dữ liệu huấn luyện và mô hình sang Google Drive!")
print("📂 Vui lòng vào Google Drive của bạn, kiểm tra thư mục: MyDrive -> FOD_YOLO_Backup_All")